In [8]:
import pdfplumber

conversations = []

with pdfplumber.open("data/parliament.pdf") as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        if text:
            conversations.append(text)

full_text = "\n".join(conversations)

# save to txt
with open("data/parliament.txt", "w") as f:
    f.write(full_text)

In [16]:
# Section headers that are navigational, not real topics
SKIP_TOPICS = {
    "THE HANSARD", "QUESTIONS AND STATEMENTS", "REQUESTS FOR STATEMENTS",
    "REQUEST FOR STATEMENT", "COMMUNICATION FROM THE CHAIR", "STATEMENTS",
    "MOTIONS", "PRAYERS", "NEXT ORDER", "BUSINESS FOR THE WEEK",
    "PAPERS", "BILLS", "NOTICES OF MOTION"
}

In [23]:
import re
import json

def extract_date(text):
    match = re.search(
        r'\b(\d{1,2}(?:st|nd|rd|th)\s+\w+\s+\d{4})\b', text
    )
    return match.group(1) if match else "Unknown Date"

def extract_parliament_info(text):
    # Matches "THIRTEENTH PARLIAMENT" etc.
    parliament = re.search(r'((?:FIRST|SECOND|THIRD|FOURTH|FIFTH|SIXTH|SEVENTH|'
                           r'EIGHTH|NINTH|TENTH|ELEVENTH|TWELFTH|THIRTEENTH|'
                           r'FOURTEENTH|FIFTEENTH)\s+PARLIAMENT)', text)
    
    # Matches "NATIONAL ASSEMBLY" or "SENATE"
    chamber = re.search(r'(NATIONAL ASSEMBLY|SENATE)', text)
    
    return {
        "parliament": parliament.group(1) if parliament else "Unknown Parliament",
        "chamber": chamber.group(1) if chamber else "Unknown Chamber"
    }

In [24]:
def is_skip_topic(topic):
    # Normalize and check if it's a generic section header
    normalized = topic.strip().upper()
    for skip in SKIP_TOPICS:
        if normalized == skip or normalized.startswith(skip + "\n"):
            return True
    return False


In [25]:
def clean_topic(topic):
    # Flatten multi-line topic headings into a single clean string
    lines = [line.strip() for line in topic.strip().splitlines() if line.strip()]
    
    # Remove generic prefix lines like "QUESTIONS AND STATEMENTS"
    filtered = [l for l in lines if l not in SKIP_TOPICS]
    
    return " ".join(filtered) if filtered else " ".join(lines)

In [26]:
def parse_hansard(text):
    chunks = []

    # Remove disclaimer lines that break up speech
    text = re.sub(
        r'Disclaimer:.*?Hansard Editor\.', '', text, flags=re.DOTALL
    )

    # Remove running headers like "30th April 2026 National Assembly Debates 2"
    text = re.sub(r'\d{1,2}\w{2}\s+\w+\s+\d{4}\s+National Assembly Debates\s+\d+', '', text)

    date = extract_date(text)
    parliament_info = extract_parliament_info(text)

    # Split on ALL-CAPS headings (2+ words, own line)
    # Heading = line(s) of ALL CAPS text, possibly spanning multiple lines
    sections = re.split(r'\n((?:[A-Z][A-Z\s,\-]+\n?){1,4})\n', text)

    current_topic = None
    for section in sections:
        stripped = section.strip()

        # Detect if this is a heading block
        if re.match(r'^[A-Z][A-Z\s,\-\n]{8,}$', stripped) and len(stripped) < 300:
            current_topic = stripped
            continue

        if not current_topic or is_skip_topic(current_topic):
            current_topic = None
            continue

         # Extract all speaker turns — greedy match to end of their speech
        turns = re.findall(
            r'(Hon\.\s+[\w\s]+(?:\([\w\s,]+\))?)\s*:\s*(.+?)(?=\nHon\.\s|\Z)',
            stripped,
            re.DOTALL
        )

        if not turns:
            current_topic = None
            continue

        topic_label = clean_topic(current_topic)
        chunk_text = f"Topic: {topic_label}\n\n"
        speakers = []

        for speaker, utterance in turns:
            speaker = speaker.strip()
            # Skip very short utterances
            utterance = utterance.strip()
            if len(utterance) < 30:
                continue
            chunk_text += f"{speaker}: {utterance}\n\n"
            speakers.append(speaker)
        
        if speakers:
            chunks.append({
                "text": chunk_text,
                "metadata": {
                    "topic": current_topic,
                    "speakers": list(set(speakers)),
                    "date": date,                           
                    "parliament": parliament_info["parliament"],
                    "chamber": parliament_info["chamber"],
                    "source": "Kenya Hansard"
            }
        })

        current_topic = None

    return chunks

In [27]:
def save_chunks(chunks, output_path="data/parsed_hansard.jsonl"):
    with open(output_path, "w", encoding="utf-8") as f:
        for chunk in chunks:
            f.write(json.dumps(chunk, ensure_ascii=False) + "\n")
    print(f"Saved {len(chunks)} chunks to {output_path}")
    for c in chunks:
        print(f" -> {c['metadata']['topic']}")

In [28]:
chunks = parse_hansard(full_text)
save_chunks(chunks)

Saved 9 chunks to data/parsed_hansard.jsonl
 -> ALLEGED ENVIRONMENTAL POLLUTION
AND DEGRADATION IN RABAI
 -> ABDUCTION AND DISAPPEARANCE OF
MR MOHAMED ABDINOOR ISMAIL
 -> UNLAWFUL DETENTION
OF A BODY BY KUTRRH
 -> PLIGHT OF RICE FARMERS IN MWEA
 -> CIRCULATION OF HARMFUL
ALCOHOLIC DRINKS
 -> DELAY IN PROCESSING OF
VALUE ADDED TAX REFUNDS
 -> OUTBREAK OF UNKNOWN ILLNESS AT BISHOP
CAVALLERA GIRLS SECONDARY SCHOOL
 -> INCREASED FATAL HIT AND RUN ACCIDENTS
ALONG KISUMU - KAKAMEGA ROAD
 -> GENDER-BASED VIOLENCE IN
MOMBASA COUNTY
